# XÂY DỰNG KHO TRI THỨC CẢI TIẾN V2 (KNOWLEDGE BASE V2 - SPO SEMANTIC GRAPH) CHO FLICKR30K
**Đề tài:** Sinh chú thích ảnh tự động tăng cường truy hồi tri thức (RAG for Image Captioning)
**Mục tiêu nâng cấp từ Kho Tri Thức V1 (`knowledge_base_level3_Flick30K`) lên Kho Tri Thức V2 (`knowledge_base_level4_Flick30K`):**
1. **Tái sử dụng 100% Vector CLIP & Chỉ mục FAISS đã chuẩn hóa (`mean_sim = 1.0000`)**: Giữ nguyên `index.faiss` và `embeddings.npy` (`145,000 × 512`, chuẩn hóa L2 từ `CLIPTextModelWithProjection`), không cần tốn hàng giờ chạy lại GPU.
2. **Làm giàu tầng Ngữ nghĩa Cấu trúc (Structured SPO Metadata)**: Trích xuất tự động **Thực thể (`objects`)**, **Thuộc tính (`attributes`)**, **Hành động (`actions`)**, **Quan hệ bộ ba (`relations` / SPO triples)** và liên kết với đồ thị cảnh **`reltr_flickr30k_full_lv5`** cho toàn bộ 145,000 câu tập Train.
3. **Khử trùng lặp ngữ nghĩa (`duplicate_of`) & Chấm điểm độ tin cậy (`confidence`, `triple_count`)**: Giúp các cấp độ **Level 2, Level 3, Level 4, Level 5** loại bỏ triệt để hiện tượng *Duplicate Token Collapse* và giảm mạnh tỷ lệ ảo giác **`CHAIR_s` / `CHAIR_i`**.

## Ghi chú phiên bản 2.1.0 (sửa lỗi bộ trích xuất SPO so với 2.0.0)

| # | Lỗi ở bản 2.0.0 | Ví dụ | Cách sửa trong 2.1.0 |
|---|---|---|---|
| 1 | Lemma danh từ sai | `bushes` → `bushe`, `men` giữ nguyên | Bảng danh từ bất quy tắc (`men→man`, `women→woman`, `children→child`, `people→person`) và quy tắc `-ches/-shes/-sses/-xes` |
| 2 | Chủ ngữ hành động luôn là `objects[0]` | `['guy','hang']` đúng nhưng `['shirt','stand','yard']` sai | Chủ ngữ là danh từ làm chủ ngữ thật (không đứng sau giới từ) |
| 3 | Quan hệ gắn nhầm danh từ gần nhất | `['hair','look','hand']` | Động từ → `(chủ ngữ, động từ, tân ngữ)`; giới từ sau động từ gắn vào chủ ngữ của động từ đó |
| 4 | Trạng từ bị coi là thực thể | `outside` nằm trong `objects` | `outside/inside` xử lý như giới từ; thêm danh sách từ bỏ qua |
| 5 | Bỏ sót động từ ngôi thứ 3, nhầm danh từ -ing thành động từ | `looks`, `plays` thành object; `building` → action `build` | Sinh dạng động từ tự động (`-s`, `-ing`), danh sách `_ING_NOUNS` |
| 6 | `smiling` bị coi là thuộc tính | `man smiling at camera` → `['camera','smiling']` | Chuyển `smiling` thành động từ; trước danh từ (`a smiling woman`) vẫn là thuộc tính |
| 7 | `'Perform'` viết hoa trong tập động từ không bao giờ khớp | | Tạo tập động từ từ danh sách gốc (toàn chữ thường) |
| 8 | **Cache cũ bị nạp lại dù code đã đổi** | Chạy lại vẫn in thống kê cũ | Kiểm tra `schema_version` trong `metadata.json`; khác `2.1.0` thì sao lưu file cũ và xây lại |

Các trường dữ liệu (`objects`, `attributes`, `actions`, `relations`, `triple_count`, `confidence`, `duplicate_of`) giữ nguyên tên và kiểu, nên code ở các level phía sau không cần sửa.

In [1]:
# ==============================================================================
# CELL 1 — KẾT NỐI GOOGLE DRIVE & KIỂM TRA KHO TRI THỨC GỐC (V1)
# ==============================================================================
import os, sys, json, time, re, shutil, unicodedata
from pathlib import Path
from collections import Counter

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Môi trường cục bộ:", e)

POSSIBLE_DATA_ROOTS = [
    Path('/content/drive/MyDrive/DataKhoaLuan/Data'),
    Path('/content/drive/MyDrive/Data/Data'),
    Path('/content/drive/MyDrive/Data'),
    Path('/content/drive/MyDrive/KhoaLuanTotNghiep/Data'),
]
DATA_ROOT = next((p for p in POSSIBLE_DATA_ROOTS if (p / 'knowledge_base_level3_Flick30K/index.faiss').exists()), Path('/content/drive/MyDrive/DataKhoaLuan/Data'))
print("✅ DATA_ROOT:", DATA_ROOT)

def retry_io(operation, attempts=4):
    for attempt in range(attempts):
        try:
            return operation()
        except OSError as exc:
            if attempt + 1 == attempts:
                raise
            time.sleep(min(2 ** attempt, 5))

def safe_write_json_drive(path, value):
    path = Path(path)
    local_tmp_dir = Path('/content/tmp_json_staging')
    local_tmp_dir.mkdir(parents=True, exist_ok=True)
    local_tmp = local_tmp_dir / f"{path.name}.{os.getpid()}.tmp"
    with open(local_tmp, 'w', encoding='utf-8') as f:
        json.dump(value, f, ensure_ascii=False)
    def _copy():
        path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(local_tmp, path)
    retry_io(_copy)
    try:
        local_tmp.unlink()
    except OSError:
        pass

def _safe_copy(src, dst):
    src, dst = Path(src), Path(dst)
    def _copy():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    retry_io(_copy)

Mounted at /content/drive
✅ DATA_ROOT: /content/drive/MyDrive/DataKhoaLuan/Data


In [2]:
# ==============================================================================
# CELL 2 — NẠP MAPPING V1 & ĐỒ THỊ CẢNH RELTR TẬP TRAIN (29,000 ẢNH)
# ==============================================================================
KB_V1_DIR = DATA_ROOT / 'knowledge_base_level3_Flick30K'
with open(KB_V1_DIR / 'mapping.json', 'r', encoding='utf-8') as f:
    kb_v1_mapping = json.load(f)
print(f"✅ Đã nạp mapping.json từ KB V1: {len(kb_v1_mapping):,} câu chú thích.")

reltr_dir = DATA_ROOT / 'reltr_flickr30k_full_lv5'
reltr_train_dict = {}
if reltr_dir.is_dir():
    single_f = reltr_dir / 'reltr_records_train.json'
    if single_f.is_file():
        with open(single_f, 'r', encoding='utf-8') as f:
            d = json.load(f)
        recs = d.get('records', d) if isinstance(d, dict) else d
        reltr_train_dict = {str(r.get('id', r.get('image_id'))): set(r.get('labels', [])) for r in recs}
    else:
        for cp in sorted(reltr_dir.glob('train_*_*.json')):
            if '.progress_' in cp.name:
                continue
            try:
                with open(cp, 'r', encoding='utf-8') as f:
                    saved = json.load(f)
                for r in saved.get('records', []):
                    reltr_train_dict[str(r.get('id', r.get('image_id')))] = set(r.get('labels', []))
            except Exception:
                pass
print(f"✅ Đã nạp nhãn đồ thị cảnh RelTR tập Train cho {len(reltr_train_dict):,} ảnh.")

# ==============================================================================
# TỰ ĐỘNG KIỂM TRA / KHỞI TẠO KHO TRI THỨC CẢI TIẾN V2 CHO FLICKR30K (~20-40 GIÂY)
# Tái sử dụng 100% index.faiss & embeddings.npy (145,000 vector 512d chuẩn L2) từ KB V1
# Bổ sung tầng ngữ nghĩa cấu trúc SPO (objects, attributes, actions, relations, duplicate_of)
# kết hợp cùng đồ thị cảnh RelTR (reltr_flickr30k_full_lv5) đã có sẵn trên Drive.
#
# PHIÊN BẢN 2.1.0 — sửa lỗi bộ trích xuất so với 2.0.0:
#   * Lemma danh từ đúng (bushes->bush, men->man, women->woman, children->child...)
#   * Chủ ngữ của hành động là danh từ làm chủ ngữ thật, không phải objects[0]
#   * Quan hệ động từ dạng (chủ ngữ, động từ, tân ngữ); giới từ gắn đúng chủ thể
#   * Nhận diện động từ ngôi thứ 3 (looks, plays...) và không coi 'building', 'clothing'... là động từ
#   * 'outside', 'inside' là giới từ/trạng từ, không còn là object
#   * Cache KB V2 cũ (2.0.0) tự bị phát hiện, sao lưu và xây lại
# ==============================================================================
import unicodedata

KB_V1_DIR = DATA_ROOT / 'knowledge_base_level3_Flick30K'
KB_V2_DIR = DATA_ROOT / 'knowledge_base_level4_Flick30K'
KB_V2_DIR.mkdir(parents=True, exist_ok=True)

SCHEMA_VERSION = '2.1.0'

_STOPWORDS_SPO = {
    'a', 'an', 'the', 'in', 'on', 'of', 'and', 'to', 'with', 'is', 'are', 'was', 'were',
    'at', 'by', 'for', 'from', 'as', 'into', 'up', 'down', 'out', 'over', 'under', 'while',
    'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight', 'nine', 'ten', 'dozen',
    'some', 'many', 'several', 'other', 'another', 'few', 'each', 'both', 'all', 'any',
    'every', 'more', 'most', 'various', 'different', 'multiple', 'numerous', 'lot', 'lots',
    'his', 'her', 'their', 'its', 'this', 'that', 'these', 'those', 'there', 'here', 'it',
    'they', 'he', 'she', 'we', 'who', 'which', 'very', 'front', 'back', 'side', 'top', 'bottom',
    'him', 'them', 'himself', 'herself', 'themselves', 'itself', 'my', 'your', 'our',
    'be', 'been', 'being', 'has', 'have', 'having', 'had', 'does', 'do', 'will', 'can',
    'could', 'would', 'not', 'but', 'or', 'so', 'if', 'then', 'when', 'where', 'what',
    'than', 'like', 'together', 'away', 'also', 'just', 'nearby', 'outdoors', 'indoors',
    'outdoor', 'indoor', 'off', 'past', 'shopping'
}

_COPULAS = {'is', 'are', 'was', 'were'}

# Từ mở đầu cụm danh từ: sau chúng, một dạng động từ thường là danh từ/tính từ ("a stand", "a smiling woman")
_NP_START = {
    'a', 'an', 'the', 'his', 'her', 'their', 'its', 'this', 'that', 'these', 'those',
    'my', 'your', 'our', 'one', 'two', 'three', 'four', 'five', 'six', 'seven', 'eight',
    'nine', 'ten', 'many', 'several', 'some', 'other', 'another', 'few', 'each', 'every'
}

_COMMON_COLORS_ATTRS = {
    'black', 'white', 'red', 'blue', 'green', 'yellow', 'brown', 'pink', 'orange', 'purple',
    'gray', 'grey', 'dark', 'light', 'bright', 'young', 'old', 'little', 'small', 'large',
    'big', 'tall', 'short', 'long', 'wooden', 'metal', 'grassy', 'snowy', 'sunny', 'wet',
    'dirty', 'clean', 'striped', 'plaid', 'happy', 'bald', 'blonde', 'blond', 'shaggy',
    'colorful', 'tan', 'silver', 'golden', 'curly'
}

# Giới từ (không gồm trạng từ 'up', 'down', 'out' vì chúng là tiểu từ của động từ)
_COMMON_PREPS = {
    'in', 'on', 'at', 'with', 'near', 'by', 'under', 'over', 'behind', 'beside', 'between',
    'through', 'across', 'into', 'onto', 'against', 'around', 'above', 'below', 'along',
    'toward', 'towards', 'beneath', 'inside', 'outside'
}

_VERB_BASES = [
    'play', 'run', 'walk', 'sit', 'stand', 'jump', 'ride', 'hold', 'wear', 'look', 'talk', 'eat',
    'drink', 'sleep', 'lay', 'lie', 'climb', 'swim', 'fly', 'carry', 'work', 'read', 'sing',
    'dance', 'fight', 'wrestle', 'skate', 'ski', 'surf', 'perform', 'watch', 'pose', 'wait',
    'throw', 'catch', 'kick', 'push', 'pull', 'cross', 'cook', 'pick', 'smile', 'laugh', 'hang',
    'lean', 'kneel', 'bend', 'pour', 'drive', 'chase', 'shake', 'hug', 'kiss', 'wave', 'point',
    'reach', 'stare', 'gaze', 'cheer', 'clap', 'shop', 'sell', 'fish', 'dig', 'mow', 'sweep',
    'serve', 'prepare', 'cut', 'chop', 'slice', 'spray', 'splash', 'dive', 'float', 'row',
    'paddle', 'sail', 'skateboard', 'snowboard', 'chat', 'speak', 'shout', 'yell', 'cry', 'grin',
    'gather', 'enjoy', 'relax', 'rest', 'travel', 'move', 'turn', 'pass', 'follow', 'lead',
    'roll', 'spin', 'balance', 'practice', 'compete', 'use', 'take', 'make', 'give', 'get',
    'pray', 'sip', 'shoot', 'lift', 'drag', 'exercise', 'jog', 'tie', 'toss', 'bounce', 'crawl',
    'pet', 'feed', 'brush', 'wash'
]
_VERB_BASES = [v for v in _VERB_BASES if v != 'shop']   # 'shopping bags' là cụm danh từ, không phải hành động

# Cụm danh từ ghép phổ biến: gộp thành một thực thể duy nhất
_COMPOUNDS = {
    ('hot', 'dog'): 'hot dog', ('ice', 'cream'): 'ice cream', ('teddy', 'bear'): 'teddy bear',
    ('fire', 'hydrant'): 'fire hydrant', ('cell', 'phone'): 'cell phone',
    ('traffic', 'light'): 'traffic light'
}
_VERB_IRREGULAR = {
    'sat': 'sit', 'ran': 'run', 'held': 'hold', 'wore': 'wear', 'ate': 'eat', 'swam': 'swim',
    'rode': 'ride', 'threw': 'throw', 'caught': 'catch'
}

def _is_cvc_single_syllable(b):
    # Gấp đôi phụ âm cuối khi thêm -ing chỉ với từ một âm tiết dạng phụ-nguyên-phụ (sit -> sitting)
    if len(b) < 3 or b[-1] in 'aeiouwxyh':
        return False
    if b[-2] not in 'aeiou' or b[-3] in 'aeiou':
        return False
    return len(re.findall(r'[aeiouy]+', b)) == 1

def _verb_forms(b):
    forms = {b}
    if b.endswith(('s', 'sh', 'ch', 'x', 'z')):
        forms.add(b + 'es')
    elif b.endswith('y') and len(b) > 2 and b[-2] not in 'aeiou':
        forms.add(b[:-1] + 'ies')
    else:
        forms.add(b + 's')
    if b.endswith('ie'):
        forms.add(b[:-2] + 'ying')
    elif b.endswith('e') and not b.endswith(('ee', 'ye', 'oe')):
        forms.add(b[:-1] + 'ing')
    elif _is_cvc_single_syllable(b):
        forms.add(b + b[-1] + 'ing')
    else:
        forms.add(b + 'ing')
    return forms

_VERB_FORM_TO_BASE = dict(_VERB_IRREGULAR)
for _b in _VERB_BASES:
    for _f in _verb_forms(_b):
        _VERB_FORM_TO_BASE.setdefault(_f, _b)

# Từ kết thúc bằng -ing nhưng là danh từ; từ kết thúc -ed là danh từ
_ING_NOUNS = {
    'building', 'clothing', 'painting', 'ceiling', 'wedding', 'morning', 'evening', 'thing',
    'something', 'anything', 'everything', 'nothing', 'string', 'spring', 'sibling', 'railing',
    'fencing', 'lighting', 'seating', 'parking', 'bedding', 'tubing', 'siding', 'stuffing',
    'drawing', 'meeting', 'ring', 'king', 'wing', 'swing', 'shopping', 'camping', 'training'
}
_ED_NOUNS = {'hundred', 'kindred', 'hatred', 'cowshed', 'bobsled'}

_IRREGULAR_PLURALS = {
    'men': 'man', 'women': 'woman', 'children': 'child', 'people': 'person', 'persons': 'person',
    'feet': 'foot', 'teeth': 'tooth', 'mice': 'mouse', 'geese': 'goose', 'oxen': 'ox',
    'leaves': 'leaf', 'knives': 'knife', 'wolves': 'wolf', 'shelves': 'shelf', 'halves': 'half',
    'lives': 'life', 'wives': 'wife', 'loaves': 'loaf', 'potatoes': 'potato', 'tomatoes': 'tomato',
    'heroes': 'hero', 'buses': 'bus', 'movies': 'movie', 'cookies': 'cookie', 'hoodies': 'hoodie',
    'selfies': 'selfie', 'brownies': 'brownie', 'smoothies': 'smoothie', 'zombies': 'zombie'
}
_PLURAL_ONLY = {
    'glasses', 'sunglasses', 'jeans', 'pants', 'shorts', 'scissors', 'clothes', 'trousers',
    'binoculars', 'goggles', 'overalls', 'tights', 'leggings', 'headphones', 'skates', 'skis'
}

def _noun_lemma(w):
    if w in _IRREGULAR_PLURALS:
        return _IRREGULAR_PLURALS[w]
    if w in _PLURAL_ONLY:
        return w
    n = len(w)
    if n > 4 and w.endswith('ies'):
        return w[:-3] + 'y'
    if n > 4 and w.endswith(('ches', 'shes', 'sses', 'xes')):
        return w[:-2]
    if n > 3 and w.endswith('s') and not w.endswith(('ss', 'us', 'is')):
        return w[:-1]
    return w

# Giữ lại tên cũ để code ở nơi khác (nếu có) vẫn gọi được
_normalize_lemma = _noun_lemma

def _ing_lemma(tok):
    base = tok[:-3]
    if len(base) >= 3 and base[-1] == base[-2] and base[-1] not in 'aeiouls':
        base = base[:-1]
    elif base.endswith(('c', 'v', 'z')):
        base += 'e'
    return base

def _verb_info(tok):
    # Trả về (lemma, is_ing) nếu tok là động từ/phân từ, ngược lại None
    base = _VERB_FORM_TO_BASE.get(tok)
    if base is not None:
        return base, tok.endswith('ing')
    if len(tok) > 5 and tok.endswith('ing') and tok not in _ING_NOUNS:
        return _ing_lemma(tok), True
    if len(tok) >= 6 and tok.endswith('ed') and tok not in _ED_NOUNS:
        return tok, False   # phân từ quá khứ (dressed, seated...): giữ nguyên dạng gốc, không lemma hoá
    return None

def extract_structured_spo_record(row_id, img_id, raw_caption, reltr_labels_set=None):
    text_clean = unicodedata.normalize('NFKC', str(raw_caption)).strip()
    raw_tokens = re.findall(r'[a-z0-9]+', text_clean.lower())
    tokens, k = [], 0
    while k < len(raw_tokens):
        pair = (raw_tokens[k], _noun_lemma(raw_tokens[k + 1])) if k + 1 < len(raw_tokens) else None
        if pair in _COMPOUNDS:
            tokens.append(_COMPOUNDS[pair])
            k += 2
        else:
            tokens.append(raw_tokens[k])
            k += 1
    n_tok = len(tokens)

    objects, attributes, actions, relations = [], [], [], []
    seen_objs, seen_attr, seen_act, seen_rel = set(), set(), set(), set()

    def _add(store, seen, item):
        key = tuple(item)
        if key not in seen:
            seen.add(key)
            store.append(item)

    subjects = []         # danh từ làm chủ ngữ chưa được động từ nào "nhận"
    last_subjects = []    # chủ ngữ của động từ gần nhất (cho "sitting and smiling")
    last_noun = None
    pending_adjs = []     # tính từ đang chờ danh từ phía sau
    pending_prep = None   # (giới từ, danh từ neo phía trước)
    pending_verb = None   # (động từ, [chủ ngữ]) đang chờ tân ngữ
    after_copula = False  # sau is/are: tính từ là vị ngữ ("the dog is black")

    i = 0
    while i < n_tok:
        tok = tokens[i]
        prev = tokens[i - 1] if i > 0 else ''

        # --- Giới từ (gồm cụm nhiều từ) ---
        pred, step = None, 1
        if tok == 'in' and tokens[i + 1:i + 3] == ['front', 'of']:
            pred, step = 'in front of', 3
        elif tok == 'on' and tokens[i + 1:i + 3] == ['top', 'of']:
            pred, step = 'on top of', 3
        elif tok == 'next' and tokens[i + 1:i + 2] == ['to']:
            pred, step = 'next to', 2
        elif tok in _COMMON_PREPS:
            pred = tok
        if pred is not None:
            anchor = pending_verb[1][-1] if pending_verb else last_noun
            pending_prep = (pred, anchor)
            pending_adjs = []
            after_copula = False
            i += step
            continue

        if tok in _STOPWORDS_SPO or len(tok) <= 2 or tok.isdigit():
            if tok in _COPULAS:
                after_copula = True
                pending_adjs = []   # tính từ đứng trước 'is/are' không còn bổ nghĩa cho danh từ phía sau
            i += 1
            continue

        # --- Thuộc tính ---
        if tok in _COMMON_COLORS_ATTRS:
            if after_copula and (subjects or last_noun):
                target = subjects[-1] if subjects else last_noun
                _add(attributes, seen_attr, [target, tok])
            else:
                pending_adjs.append(tok)
            i += 1
            continue

        # --- Động từ / phân từ ---
        vinfo = _verb_info(tok)
        if vinfo is not None:
            vlem, is_ing = vinfo
            in_np = prev in _NP_START or bool(pending_adjs)
            if in_np and (is_ing or tok.endswith('ed')):
                # "a smiling woman", "a tattooed arm": đóng vai tính từ
                pending_adjs.append(tok)
                i += 1
                continue
            if not is_ing and not tok.endswith('ed') and (in_np or pending_verb is not None or not subjects):
                # Dạng gốc / ngôi thứ 3 chỉ là động từ khi đang có chủ ngữ chờ động từ
                # ("woman plays"); còn lại là danh từ ("a stand", "catching fish", "hot dog stand")
                vinfo = None
        if vinfo is not None:
            vlem, _ = vinfo
            subs = subjects[-2:] or last_subjects[-2:] or ['person']
            for s in subs:
                _add(actions, seen_act, [s, vlem])
            if subjects:
                last_subjects = subjects[-2:]
                subjects = []
            pending_verb = (vlem, subs)
            pending_prep = None
            pending_adjs = []
            after_copula = False
            i += 1
            continue

        # --- Danh từ ---
        lem = _noun_lemma(tok)
        if len(lem) < 3:
            i += 1
            continue
        if lem not in seen_objs:
            seen_objs.add(lem)
            objects.append(lem)
        for adj in pending_adjs:
            _add(attributes, seen_attr, [lem, adj])
        pending_adjs = []
        after_copula = False
        if pending_prep is not None:
            p, anchor = pending_prep
            if anchor is not None and anchor != lem:
                _add(relations, seen_rel, [anchor, p, lem])
            pending_prep = None
            pending_verb = None
        elif pending_verb is not None:
            v, subs = pending_verb
            for s in subs:
                if s != lem:
                    _add(relations, seen_rel, [s, v, lem])
            pending_verb = None
        else:
            subjects.append(lem)
        last_noun = lem
        i += 1

    # Bổ sung thực thể từ nhãn RelTR của ảnh nguồn (chỉ nhận từ có trong caption)
    if reltr_labels_set:
        caption_lemmas = {_noun_lemma(t) for t in tokens}
        for rl in reltr_labels_set:
            for part in str(rl).lower().split():
                nh = _noun_lemma(part)
                if (nh in caption_lemmas and nh not in seen_objs and len(nh) >= 3
                        and nh not in _STOPWORDS_SPO and nh not in _COMMON_COLORS_ATTRS
                        and nh not in _COMMON_PREPS and nh not in _VERB_FORM_TO_BASE):
                    seen_objs.add(nh)
                    objects.append(nh)

    triple_count = len(relations) + len(actions) + len(attributes)
    confidence = round(min(0.98, 0.65 + 0.05 * len(objects) + 0.04 * triple_count), 4)
    return {
        'vector_id': int(row_id),
        'row_id': int(row_id),
        'image_id': img_id,
        'caption': text_clean,
        'split': 'train',
        'objects': objects[:8],
        'attributes': attributes[:6],
        'actions': actions[:5],
        'relations': relations[:6],
        'triple_count': int(triple_count),
        'confidence': float(confidence),
        'duplicate_of': None
    }

def ensure_flickr30k_kb_v2(kb_v1_mapping, reltr_train_dict=None, force_rebuild=False):
    v2_map_path = KB_V2_DIR / 'mapping_v2_spo.json'
    v2_meta_path = KB_V2_DIR / 'metadata.json'
    v2_index_path = KB_V2_DIR / 'index.faiss'
    v2_emb_path = KB_V2_DIR / 'embeddings.npy'

    # Đồng bộ index.faiss và embeddings.npy sang thư mục KB V2 nếu chưa có
    if not v2_index_path.is_file() and (KB_V1_DIR / 'index.faiss').is_file():
        _safe_copy(KB_V1_DIR / 'index.faiss', v2_index_path)
    if not v2_emb_path.is_file() and (KB_V1_DIR / 'embeddings.npy').is_file():
        _safe_copy(KB_V1_DIR / 'embeddings.npy', v2_emb_path)

    old_version = None
    if v2_map_path.is_file():
        try:
            with open(v2_meta_path, 'r', encoding='utf-8') as f:
                old_version = json.load(f).get('schema_version')
        except Exception:
            old_version = None
        if old_version == SCHEMA_VERSION and not force_rebuild:
            try:
                with open(v2_map_path, 'r', encoding='utf-8') as f:
                    records_v2 = json.load(f)
                if len(records_v2) == len(kb_v1_mapping) and 'objects' in records_v2[0]:
                    print(f"✅ Đã nạp Kho Tri Thức Cải Tiến V2 (knowledge_base_level4_Flick30K, schema {SCHEMA_VERSION}): {len(records_v2):,} bản ghi ngữ nghĩa SPO!")
                    return records_v2
            except Exception as e:
                print(f"⚠️ File mapping_v2_spo.json cần tạo lại: {e}")
        else:
            tag = re.sub(r'[^0-9A-Za-z.]+', '_', str(old_version or 'unknown'))
            bak_path = KB_V2_DIR / f'mapping_v2_spo.bak_schema_{tag}.json'
            if not bak_path.is_file():
                _safe_copy(v2_map_path, bak_path)
                print(f"♻️ Phát hiện KB V2 phiên bản cũ ({old_version or 'không rõ'}). Đã sao lưu sang {bak_path.name} và sẽ xây lại với schema {SCHEMA_VERSION}.")

    print("🚀 Đang nâng cấp Kho Tri Thức Flickr30k V1 -> V2 (Trích xuất Objects, Attributes, Actions, SPO Relations & Khử trùng lặp cho 145,000 câu)...")
    t0 = time.time()
    records_v2 = []
    seen_norm_text = {}
    dup_count = 0

    for i in range(len(kb_v1_mapping)):
        entry = kb_v1_mapping[i] if isinstance(kb_v1_mapping, list) else kb_v1_mapping[str(i)]
        img_id = entry['image_id']
        cap = entry['caption']
        rl_set = reltr_train_dict.get(str(img_id), set()) if reltr_train_dict else None
        rec = extract_structured_spo_record(i, img_id, cap, rl_set)

        norm_key = re.sub(r'[^a-z0-9]+', ' ', cap.lower()).strip()
        if norm_key in seen_norm_text:
            rec['duplicate_of'] = seen_norm_text[norm_key]
            dup_count += 1
        else:
            seen_norm_text[norm_key] = i
        records_v2.append(rec)

    safe_write_json_drive(v2_map_path, records_v2)
    meta_v2 = {
        'dataset': 'Flickr30k',
        'schema_version': SCHEMA_VERSION,
        'num_captions': len(records_v2),
        'embedding_dim': 512,
        'encoder_model': 'openai/clip-vit-base-patch32 (with text_projection, L2-normalized)',
        'index_type': 'IndexFlatIP',
        'duplicates_flagged': dup_count,
        'semantic_fields': ['objects', 'attributes', 'actions', 'relations', 'triple_count', 'confidence', 'duplicate_of'],
        'created_at': time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())
    }
    safe_write_json_drive(v2_meta_path, meta_v2)
    print(f"🎉 Hoàn tất xây dựng Kho Tri Thức V2 Flickr30k (schema {SCHEMA_VERSION}) trong {time.time()-t0:.1f}s! (Phát hiện & đánh dấu {dup_count:,} câu trùng lặp).")
    return records_v2

kb_v2_records = ensure_flickr30k_kb_v2(kb_v1_mapping, reltr_train_dict)


✅ Đã nạp mapping.json từ KB V1: 145,000 câu chú thích.
✅ Đã nạp nhãn đồ thị cảnh RelTR tập Train cho 29,000 ảnh.
♻️ Phát hiện KB V2 phiên bản cũ (2.0.0). Đã sao lưu sang mapping_v2_spo.bak_schema_2.0.0.json và sẽ xây lại với schema 2.1.0.
🚀 Đang nâng cấp Kho Tri Thức Flickr30k V1 -> V2 (Trích xuất Objects, Attributes, Actions, SPO Relations & Khử trùng lặp cho 145,000 câu)...
🎉 Hoàn tất xây dựng Kho Tri Thức V2 Flickr30k (schema 2.1.0) trong 13.9s! (Phát hiện & đánh dấu 524 câu trùng lặp).


In [3]:
# ==============================================================================
# CELL 3 — THỐNG KÊ ĐỐI CHIẾU KHO TRI THỨC V1 vs KHO TRI THỨC V2 & MẪU MINH CHỨNG
# ==============================================================================
total_caps = len(kb_v2_records)
with_objs = sum(1 for r in kb_v2_records if len(r['objects']) > 0)
with_attrs = sum(1 for r in kb_v2_records if len(r['attributes']) > 0)
with_acts = sum(1 for r in kb_v2_records if len(r['actions']) > 0)
with_rels = sum(1 for r in kb_v2_records if len(r['relations']) > 0)
dups = sum(1 for r in kb_v2_records if r['duplicate_of'] is not None)

print("=" * 88)
print("📊 BẢNG ĐỐI CHIẾU CẤU TRÚC KHO TRI THỨC FLICKR30K: BẢN CŨ (KB V1) vs BẢN CẢI TIẾN (KB V2)")
print("=" * 88)
print(f"| {'Tiêu chí kỹ thuật':<36} | {'Kho Cũ (KB V1)':<20} | {'Kho Cải Tiến (KB V2)':<22} |")
print(f"| {'-'*36} | {'-'*20} | {'-'*22} |")
print(f"| {'Tổng số câu chú thích (Train)':<36} | {total_caps:<20,d} | {total_caps:<22,d} |")
print(f"| {'Không gian Vector CLIP ViT-B/32':<36} | {'512d L2-normalized':<20} | {'512d L2-normalized':<22} |")
print(f"| {'Số câu có Thực thể (Objects)':<36} | {'0 (Chỉ lưu text thô)':<20} | {f'{with_objs:,d} ({with_objs*100/total_caps:.1f}%)':<22} |")
print(f"| {'Số câu có Thuộc tính (Attributes)':<36} | {'0 (Không trích xuất)':<20} | {f'{with_attrs:,d} ({with_attrs*100/total_caps:.1f}%)':<22} |")
print(f"| {'Số câu có Hành động (Actions)':<36} | {'0 (Không trích xuất)':<20} | {f'{with_acts:,d} ({with_acts*100/total_caps:.1f}%)':<22} |")
print(f"| {'Số câu có Quan hệ SPO (Relations)':<36} | {'0 (Không trích xuất)':<20} | {f'{with_rels:,d} ({with_rels*100/total_caps:.1f}%)':<22} |")
print(f"| {'Cờ khử câu trùng lặp (duplicate_of)':<36} | {'Không có':<20} | {f'{dups:,d} câu được gắn cờ':<22} |")
print("=" * 88)

print("\n🔍 3 BẢN GHI MẪU TRONG KHO TRI THỨC CẢI TIẾN V2 (knowledge_base_level4_Flick30K):")
for idx_s in range(3):
    r = kb_v2_records[idx_s]
    print(f"\n[Row #{r['row_id']} | Image ID: {r['image_id']} | Confidence: {r['confidence']}]")
    print(f"  * Caption   : {r['caption']}")
    print(f"  * Objects   : {r['objects']}")
    print(f"  * Attributes: {r['attributes']}")
    print(f"  * Actions   : {r['actions']}")
    print(f"  * Relations : {r['relations']}")

📊 BẢNG ĐỐI CHIẾU CẤU TRÚC KHO TRI THỨC FLICKR30K: BẢN CŨ (KB V1) vs BẢN CẢI TIẾN (KB V2)
| Tiêu chí kỹ thuật                    | Kho Cũ (KB V1)       | Kho Cải Tiến (KB V2)   |
| ------------------------------------ | -------------------- | ---------------------- |
| Tổng số câu chú thích (Train)        | 145,000              | 145,000                |
| Không gian Vector CLIP ViT-B/32      | 512d L2-normalized   | 512d L2-normalized     |
| Số câu có Thực thể (Objects)         | 0 (Chỉ lưu text thô) | 144,996 (100.0%)       |
| Số câu có Thuộc tính (Attributes)    | 0 (Không trích xuất) | 71,176 (49.1%)         |
| Số câu có Hành động (Actions)        | 0 (Không trích xuất) | 128,571 (88.7%)        |
| Số câu có Quan hệ SPO (Relations)    | 0 (Không trích xuất) | 139,714 (96.4%)        |
| Cờ khử câu trùng lặp (duplicate_of)  | Không có             | 524 câu được gắn cờ    |

🔍 3 BẢN GHI MẪU TRONG KHO TRI THỨC CẢI TIẾN V2 (knowledge_base_level4_Flick30K):

[Row #0 | Image ID: 1000092

In [4]:
# ==============================================================================
# CELL 4 — TỰ KIỂM TRA BỘ TRÍCH XUẤT TRÊN CÁC CÂU MẪU (KHÔNG CẦN DỮ LIỆU DRIVE)
# ==============================================================================
_test_caps = [
    "Two young guys with shaggy hair look at their hands while hanging out in the yard.",
    "Two young, White males are outside near many bushes.",
    "Two men in green shirts are standing in a yard.",
    "A man in a blue shirt is standing in front of a building.",
    "A woman plays guitar and a man sings.",
    "A young smiling woman sits next to a dog.",
]
for _c in _test_caps:
    _r = extract_structured_spo_record(0, 'test', _c, None)
    print(f"\n{_c}")
    for _k in ('objects', 'attributes', 'actions', 'relations'):
        print(f"   {_k:<10}: {_r[_k]}")


Two young guys with shaggy hair look at their hands while hanging out in the yard.
   objects   : ['guy', 'hair', 'hand', 'yard']
   attributes: [['guy', 'young'], ['hair', 'shaggy']]
   actions   : [['guy', 'look'], ['guy', 'hang']]
   relations : [['guy', 'with', 'hair'], ['guy', 'at', 'hand'], ['guy', 'in', 'yard']]

Two young, White males are outside near many bushes.
   objects   : ['male', 'bush']
   attributes: [['male', 'young'], ['male', 'white']]
   actions   : []
   relations : [['male', 'near', 'bush']]

Two men in green shirts are standing in a yard.
   objects   : ['man', 'shirt', 'yard']
   attributes: [['shirt', 'green']]
   actions   : [['man', 'stand']]
   relations : [['man', 'in', 'shirt'], ['man', 'in', 'yard']]

A man in a blue shirt is standing in front of a building.
   objects   : ['man', 'shirt', 'building']
   attributes: [['shirt', 'blue']]
   actions   : [['man', 'stand']]
   relations : [['man', 'in', 'shirt'], ['man', 'in front of', 'building']]

A woman